### Why a process needs to store both context and trapframe?

say user process A do syscall to read file.

First, kernel needs to store process A's context.

Now, kernel is doing the work for process A, but kernel could also be cutoff during this time, so kernel also need to track its context for the work it's doing for process A.

- trapframe: process A's context. Things needed to resume A.
- context: kernel's context when it's doing work for A. Things needed to resume kernel finishing work for A.

---
### When user process trap happens, does it directyly execute kernel code?

No, it firsts needs to go to trampoline to save registers, load kernel page table etc, then go to `usertrap` to execute kernel code. Those prep work is necessary.

Can we hardcode address of `usertrap` in trampoline?

In thoery, yes. It's author's design choice to store it in trapframe.

---
### How many times a process will enter forkret()?

Only 1 time!

`ra` is **where it's stopped last time**, in allocproc, since process hasn't run yet, it doesn't have **where it's stopped last time**. 

```c
p->context.ra = (uint64)forkret;
p->context.sp = p->kstack + PGSIZE;
```

later, when the process runs, `p->context.ra` will be override.

When a process starts, by `exec()` or `fork()`, looking for `p->trapframe->epc`, it will be copied from parent or from ELF.